# SegFormer Fine-tuning — Parking Segmentation
**Runtime:** T4 GPU | **Dataset:** combined_dataset_segmentation (522 images, binary: background vs parkable)

In [ ]:
# ============================================================
# Cell 1 — Imports & Preparation
# ============================================================

!pip install -q transformers gdown

import json
import numpy as np
import torch
import torch.nn as nn
from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from transformers import AutoImageProcessor, SegformerForSemanticSegmentation, get_scheduler
from tqdm.auto import tqdm
import zipfile, os

# ── Download dataset from Google Drive ─────────────────────
# Share your zip as "Anyone with the link" and paste the file ID here
# Example: if link is https://drive.google.com/file/d/1AbCdEfGh/view → ID = "1AbCdEfGh"
GDRIVE_FILE_ID = "YOUR_FILE_ID_HERE"  # <-- change this

import gdown
zip_path = '/content/combined_dataset_segmentation.zip'
gdown.download(id=GDRIVE_FILE_ID, output=zip_path, quiet=False)

with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall('/content/')

# Auto-detect extracted folder
DATA_DIR = Path('/content/combined_dataset_segmentation')
if not DATA_DIR.exists():
    candidates = [p for p in Path('/content').iterdir() if p.is_dir() and (p / 'train.txt').exists()]
    DATA_DIR = candidates[0] if candidates else DATA_DIR

print(f"Dataset: {DATA_DIR}")
print(f"  train: {len(open(DATA_DIR / 'train.txt').readlines())}")
print(f"  val:   {len(open(DATA_DIR / 'val.txt').readlines())}")
print(f"  test:  {len(open(DATA_DIR / 'test.txt').readlines())}")

# Config
BACKBONE = 'b0'  # change to b2, b3, etc.
MODEL_ID = f'nvidia/segformer-{BACKBONE}-finetuned-cityscapes-1024-1024'
NUM_CLASSES = 2
ID2LABEL = {0: 'background', 1: 'parkable'}
LABEL2ID = {'background': 0, 'parkable': 1}
IMG_SIZE = 512
BATCH_SIZE = 8
EPOCHS = 30
LR = 6e-5

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device} — {torch.cuda.get_device_name(0) if device == 'cuda' else 'CPU'}")


# Dataset class
class ParkingDataset(Dataset):
    def __init__(self, split, processor):
        with open(DATA_DIR / f'{split}.txt') as f:
            self.ids = [l.strip() for l in f if l.strip()]
        self.processor = processor

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        sid = self.ids[idx]
        img_path = DATA_DIR / 'images' / f'{sid}.png'
        if not img_path.exists():
            img_path = DATA_DIR / 'images' / f'{sid}.jpg'
        mask_path = DATA_DIR / 'masks' / f'{sid}.png'
        if not mask_path.exists():
            mask_path = DATA_DIR / 'masks' / f'{sid}.jpg'

        image = Image.open(img_path).convert('RGB').resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR)
        mask = Image.open(mask_path).convert('L').resize((IMG_SIZE, IMG_SIZE), Image.NEAREST)

        mask_np = np.array(mask, dtype=np.int64)
        mask_np[mask_np == 255] = 1

        pixel_values = self.processor(images=image, return_tensors='pt')['pixel_values'].squeeze(0)
        labels = torch.tensor(mask_np, dtype=torch.long)
        return pixel_values, labels


# Load processor & model
processor = AutoImageProcessor.from_pretrained(MODEL_ID)
model = SegformerForSemanticSegmentation.from_pretrained(
    MODEL_ID,
    num_labels=NUM_CLASSES,
    id2label=ID2LABEL,
    label2id=LABEL2ID,
    ignore_mismatched_sizes=True,
).to(device)

# Dataloaders
train_loader = DataLoader(ParkingDataset('train', processor), batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(ParkingDataset('val', processor), batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(ParkingDataset('test', processor), batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"\nModel: {MODEL_ID}")
print(f"Train: {len(train_loader.dataset)} | Val: {len(val_loader.dataset)} | Test: {len(test_loader.dataset)}")
print("Ready!")

In [ ]:
# ============================================================
# Cell 2 — Fine-tuning
# ============================================================

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
num_steps = len(train_loader) * EPOCHS
lr_scheduler = get_scheduler('cosine', optimizer=optimizer,
                             num_warmup_steps=int(0.1 * num_steps),
                             num_training_steps=num_steps)

best_miou = 0.0
history = []

for epoch in range(1, EPOCHS + 1):
    # --- Train ---
    model.train()
    train_loss = 0.0
    pbar = tqdm(train_loader, desc=f'Epoch {epoch}/{EPOCHS} [train]')
    for px, lb in pbar:
        px, lb = px.to(device), lb.to(device)
        logits = model(pixel_values=px).logits
        up = nn.functional.interpolate(logits, size=lb.shape[1:], mode='bilinear', align_corners=False)
        loss = nn.functional.cross_entropy(up, lb, ignore_index=255)
        loss.backward()
        optimizer.step()
        lr_scheduler.step()
        optimizer.zero_grad()
        train_loss += loss.item()
        pbar.set_postfix(loss=f'{loss.item():.4f}')

    # --- Val ---
    model.eval()
    val_loss = 0.0
    ious_bg, ious_pk = [], []
    with torch.no_grad():
        for px, lb in val_loader:
            px, lb = px.to(device), lb.to(device)
            logits = model(pixel_values=px).logits
            up = nn.functional.interpolate(logits, size=lb.shape[1:], mode='bilinear', align_corners=False)
            val_loss += nn.functional.cross_entropy(up, lb, ignore_index=255).item()
            preds = up.argmax(dim=1)
            for cls, store in [(0, ious_bg), (1, ious_pk)]:
                inter = ((preds == cls) & (lb == cls)).sum().item()
                union = ((preds == cls) | (lb == cls)).sum().item()
                if union > 0:
                    store.append(inter / union)

    bg = np.mean(ious_bg) if ious_bg else 0
    pk = np.mean(ious_pk) if ious_pk else 0
    miou = (bg + pk) / 2
    avg_tl = train_loss / len(train_loader)
    avg_vl = val_loss / len(val_loader)

    print(f'  train_loss={avg_tl:.4f} | val_loss={avg_vl:.4f} | bg_IoU={bg:.4f} | park_IoU={pk:.4f} | mIoU={miou:.4f}')
    history.append({'epoch': epoch, 'train_loss': avg_tl, 'val_loss': avg_vl, 'bg_iou': bg, 'parkable_iou': pk, 'miou': miou})

    if miou > best_miou:
        best_miou = miou
        model.save_pretrained(f'/content/segformer-{BACKBONE}-parkable-best')
        processor.save_pretrained(f'/content/segformer-{BACKBONE}-parkable-best')
        print(f'  -> Best model saved (mIoU={miou:.4f})')

# Save last + history
model.save_pretrained(f'/content/segformer-{BACKBONE}-parkable-last')
processor.save_pretrained(f'/content/segformer-{BACKBONE}-parkable-last')
with open(f'/content/history_{BACKBONE}.json', 'w') as f:
    json.dump(history, f, indent=2)

print(f'\nDone! Best mIoU: {best_miou:.4f}')

In [ ]:
# ============================================================
# Cell 3 — Validation (test set evaluation)
# ============================================================

import matplotlib.pyplot as plt

# Load best model
best_model = SegformerForSemanticSegmentation.from_pretrained(f'/content/segformer-{BACKBONE}-parkable-best').to(device)
best_model.eval()

test_ious_bg, test_ious_pk = [], []
with torch.no_grad():
    for px, lb in tqdm(test_loader, desc='Test'):
        px, lb = px.to(device), lb.to(device)
        logits = best_model(pixel_values=px).logits
        up = nn.functional.interpolate(logits, size=lb.shape[1:], mode='bilinear', align_corners=False)
        preds = up.argmax(dim=1)
        for cls, store in [(0, test_ious_bg), (1, test_ious_pk)]:
            inter = ((preds == cls) & (lb == cls)).sum().item()
            union = ((preds == cls) | (lb == cls)).sum().item()
            if union > 0:
                store.append(inter / union)

bg = np.mean(test_ious_bg)
pk = np.mean(test_ious_pk)
miou = (bg + pk) / 2
print(f'Test — bg_IoU={bg:.4f} | parkable_IoU={pk:.4f} | mIoU={miou:.4f}')

# Plot training curves
with open(f'/content/history_{BACKBONE}.json') as f:
    h = json.load(f)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
epochs = [x['epoch'] for x in h]
ax1.plot(epochs, [x['train_loss'] for x in h], label='train')
ax1.plot(epochs, [x['val_loss'] for x in h], label='val')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss'); ax1.legend(); ax1.set_title('Loss')

ax2.plot(epochs, [x['miou'] for x in h], label='mIoU')
ax2.plot(epochs, [x['bg_iou'] for x in h], label='bg IoU', linestyle='--')
ax2.plot(epochs, [x['parkable_iou'] for x in h], label='parkable IoU', linestyle='--')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('IoU'); ax2.legend(); ax2.set_title('IoU')
plt.tight_layout()
plt.savefig('/content/training_curves.png', dpi=150)
plt.show()

In [ ]:
# ============================================================
# Cell 4 — Inference & Download
# ============================================================

import matplotlib.pyplot as plt

PALETTE = np.array([[0, 0, 0], [0, 255, 128]], dtype=np.uint8)  # black=bg, green=parkable

# Run on a few test images
test_ids = open(DATA_DIR / 'test.txt').read().strip().split('\n')[:6]

fig, axes = plt.subplots(len(test_ids), 3, figsize=(15, 5 * len(test_ids)))
if len(test_ids) == 1:
    axes = [axes]

os.makedirs('/content/predictions', exist_ok=True)

best_model.eval()
for i, sid in enumerate(test_ids):
    img_path = DATA_DIR / 'images' / f'{sid}.png'
    if not img_path.exists():
        img_path = DATA_DIR / 'images' / f'{sid}.jpg'
    mask_path = DATA_DIR / 'masks' / f'{sid}.png'
    if not mask_path.exists():
        mask_path = DATA_DIR / 'masks' / f'{sid}.jpg'

    image = Image.open(img_path).convert('RGB')
    gt_mask = np.array(Image.open(mask_path).convert('L'))
    gt_mask[gt_mask == 255] = 1
    orig_size = image.size  # (W, H)

    inputs = processor(images=image, return_tensors='pt').to(device)
    with torch.no_grad():
        logits = best_model(**inputs).logits
        up = nn.functional.interpolate(logits, size=(orig_size[1], orig_size[0]), mode='bilinear', align_corners=False)
        pred = up.argmax(dim=1).squeeze().cpu().numpy()

    # Color maps
    pred_color = PALETTE[pred]
    gt_color = PALETTE[gt_mask]

    # Save
    Image.fromarray(pred_color).save(f'/content/predictions/{sid}_pred.png')
    overlay = Image.blend(image.resize((pred_color.shape[1], pred_color.shape[0])),
                          Image.fromarray(pred_color), alpha=0.4)
    overlay.save(f'/content/predictions/{sid}_overlay.png')

    axes[i][0].imshow(image); axes[i][0].set_title('Image'); axes[i][0].axis('off')
    axes[i][1].imshow(gt_color); axes[i][1].set_title('Ground Truth'); axes[i][1].axis('off')
    axes[i][2].imshow(pred_color); axes[i][2].set_title('Prediction'); axes[i][2].axis('off')

plt.tight_layout()
plt.savefig('/content/predictions/comparison.png', dpi=150)
plt.show()

# Zip best model + predictions for download
!cd /content && zip -r segformer_parkable_results.zip \
    segformer-{BACKBONE}-parkable-best/ \
    predictions/ \
    history_{BACKBONE}.json \
    training_curves.png

files.download('/content/segformer_parkable_results.zip')
print('Download started!')

In [ ]:
# ============================================================
# Cell 5 — Zip & Download best + last checkpoints
# ============================================================

import shutil
from google.colab import files

zip_name = f'/content/segformer-{BACKBONE}-parkable-checkpoints'
shutil.make_archive(zip_name, 'zip', '/content/', f'segformer-{BACKBONE}-parkable-best')

# Add -last to the same zip
import zipfile
with zipfile.ZipFile(f'{zip_name}.zip', 'a') as z:
    last_dir = Path(f'/content/segformer-{BACKBONE}-parkable-last')
    for f in last_dir.rglob('*'):
        z.write(f, f'segformer-{BACKBONE}-parkable-last/{f.relative_to(last_dir)}')

# Also add history
with zipfile.ZipFile(f'{zip_name}.zip', 'a') as z:
    z.write(f'/content/history_{BACKBONE}.json', f'history_{BACKBONE}.json')

print(f'Zip: {zip_name}.zip')
!ls -lh {zip_name}.zip
files.download(f'{zip_name}.zip')